# Notebook 1: Preprocessing & Segmentation

This notebook walks you through the first stage of the `liv_zones` pipeline: loading a raw microscopy image, running segmentation to identify cells and organelles, and computing distance maps that encode each object's position within the hepatic acinus.

By the end of this notebook you will have produced:
- **Instance segmentation masks** for cells, mitochondria, lipid droplets, and peroxisomes
- **Distance maps** from the central vein, portal vein, and cell boundaries

These files are the inputs for Notebook 2 (Feature Extraction).

---

## Biological background

The liver is divided into repeating functional units called **hepatic acini**. Each acinus spans the distance between a **portal vein** (bringing nutrient-rich blood from the gut) and a **central vein** (draining blood toward the heart). Hepatocytes (liver cells) along this axis are exposed to different oxygen and nutrient concentrations, causing them to specialize — a phenomenon called **liver zonation**.

`liv_zones` quantifies this zonation by measuring organelle properties at precise positions along the portal-to-central axis.

---

> **GPU note:** Segmentation (Section 4) requires a CUDA-capable GPU and the pretrained Cellpose models. If you do not have a GPU available, **skip Section 4** and use the pre-computed masks provided in `sample_data/output/` to continue with the rest of this notebook and with Notebook 2.

## 1. Setup

Import the packages we need and define file paths. Adjust `IMAGE_PATH` and `SAVE_PATH` to point at your own data when working with real images.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

import tifffile

from liv_zones.preprocess import preprocessing

# ── Paths ──────────────────────────────────────────────────────────────────
# Directory containing the per-channel TIF files.
DATA_DIR  = Path("/path/to/your/data")
SAVE_PATH = Path("sample_data/output")

SAVE_PATH.mkdir(parents=True, exist_ok=True)

# ── Z-slice ────────────────────────────────────────────────────────────────
# Set this to match the z-slice chosen in Notebook 0.
Z_SLICE = 25   # e.g. 25 → loads files ending in --Z25--C??.tif

def channel_path(z: int, c: int) -> Path:
    """Return the path for a given z-slice and channel index."""
    suffix = f"--Z{z:02d}--C{c:02d}.tif"
    matches = list(DATA_DIR.glob(f"*{suffix}"))
    if not matches:
        raise FileNotFoundError(f"No file matching *{suffix} in {DATA_DIR}")
    return matches[0]

# ── Scale ──────────────────────────────────────────────────────────────────
# The number of pixels per micron in your image.
SCALE = 22 

print(f"Data dir: {DATA_DIR}")
print(f"Output:   {SAVE_PATH}")
print(f"Z-slice:  Z{Z_SLICE:02d}")
print(f"Scale:    {SCALE} px/µm")

## 2. Loading the input channels

The raw data is stored as separate TIF files — one per channel per z-slice, named:

```
Region 1_Merged--Z{zz}--C{cc}.tif
```

Each channel was acquired with a different dye:

| File channel | Structure | Variable |
|:---:|:---|:---|
| C00 | Mitochondria | `ch_mito` |
| C01 | Actin | `ch_actin` |
| C02 | Lipid droplets | `ch_lipid` |
| C03 | Peroxisomes | `ch_perox` |

We load them individually and stack into a `(4, H, W)` array in the order expected by `preprocessing()`:
actin → mito → lipid → perox.

In [ ]:
ch_mito  = tifffile.imread(str(channel_path(Z_SLICE, 0)))
ch_actin = tifffile.imread(str(channel_path(Z_SLICE, 1)))
ch_lipid = tifffile.imread(str(channel_path(Z_SLICE, 2)))
ch_perox = tifffile.imread(str(channel_path(Z_SLICE, 3)))

# Stack in the order preprocessing() expects: actin, mito, lipid, perox
image = np.stack([ch_actin, ch_mito, ch_lipid, ch_perox], axis=0)

print(f"Loaded z-slice Z{Z_SLICE:02d}")
print(f"Image shape: {image.shape}  →  (channels, height, width)")
print(f"Data type:   {image.dtype}")

## 2b. Crop to a single acinus

Whole-lobe images contain many acini. Provide the pixel coordinates of the **central vein** and **portal vein** centers — `dispCrop()` will automatically size and rotate the crop rectangle (1.2× the vein-to-vein distance wide, 0.4× tall).

**To find coordinates in FIJI:** open the channel TIF, hover over each vein center, and read the `x, y` values from the status bar.

Two options are provided below: set coordinates manually (Option A) or pick them by clicking on a downsampled preview (Option B).

In [ ]:
# ── Option A: set coordinates manually ────────────────────────────────────
# Open the image in FIJI (Image > Show Info gives pixel coordinates on hover).
# Enter the x, y pixel coordinates of each vein center below.

CV_coords = np.array([7648, 10432])  # central vein  (x, y)
PV_coords = np.array([13952, 4000])  # portal vein   (x, y)

# ── Option B: click on a downsampled preview ───────────────────────────────
# Uncomment the block below to pick coordinates interactively.
# Coordinates are automatically scaled back to full resolution.

# DOWNSAMPLE = 4  # display every Nth pixel — increase if still too slow
# %matplotlib widget
# fig, ax = plt.subplots(figsize=(10, 10))
# preview = ch_actin[::DOWNSAMPLE, ::DOWNSAMPLE]
# vmin, vmax = np.percentile(preview, [1, 99])
# ax.imshow(preview, cmap="gray", vmin=vmin, vmax=vmax)
# ax.set_title(
#     f"Image downsampled {DOWNSAMPLE}×  —  "
#     "click (1) central vein, (2) portal vein, then press Enter",
#     fontsize=10,
# )
# plt.tight_layout()
# coords = plt.ginput(2, timeout=0)
# plt.close()
# CV_coords = np.array(coords[0]) * DOWNSAMPLE
# PV_coords = np.array(coords[1]) * DOWNSAMPLE

print(f"Central vein : x={CV_coords[0]:.1f}, y={CV_coords[1]:.1f}")
print(f"Portal vein  : x={PV_coords[0]:.1f}, y={PV_coords[1]:.1f}")

In [ ]:
from PIL import Image as PILImage
from liv_zones.crop import dispCrop

# Apply the same crop to all 4 channels.
# crop_info is computed from CV/PV coords on the first channel and reused for the rest.
crop_info = None
cropped = []
for arr in [ch_actin, ch_mito, ch_lipid, ch_perox]:
    pil_img = PILImage.fromarray(arr)
    cropped_arr, crop_info = dispCrop(pil_img, CV_coords=CV_coords, PV_coords=PV_coords, crop_info=crop_info)
    cropped.append(cropped_arr)

image = np.stack(cropped, axis=0)
print(f"Cropped image shape: {image.shape}  →  (channels, height, width)")

In [ ]:
channel_names = ["Actin (C01)", "Mitochondria (C00)", "Lipid droplets (C02)", "Peroxisomes (C03)"]
cmaps = ["Greens", "Reds", "Blues", "Oranges"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (ax, name, cmap) in enumerate(zip(axes, channel_names, cmaps)):
    ch = image[i]
    vmin, vmax = np.percentile(ch, [1, 99])
    ax.imshow(ch, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(name, fontsize=10)
    ax.axis("off")

plt.suptitle("Raw image — one channel per panel", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 3. Running preprocessing

The `preprocessing()` function does three things in sequence:

1. **Segments each organelle** using pretrained Cellpose models. It outputs a labeled *instance segmentation mask* — an array of integers where each unique nonzero value identifies one distinct organelle.
2. **Computes vein distance maps** using the cell mask geometry to find the two vein regions, then runs a Euclidean distance transform from each vein.
3. **Computes the cell-boundary distance map** — how far each pixel is from the nearest cell boundary.

### Arguments

| Argument | Description |
|:---|:---|
| `image_path` | Path to the multi-channel TIF |
| `save_path` | Directory where all output `.npy` files are written |
| `channels` | Dict mapping channel name → channel index in the TIF |
| `feature_list` | List of features to compute. If `None`, automatically detects what's missing. |



**Important:** If your channel order is different, update the dictionary below to match.

In [ ]:
# Pass the cropped arrays directly so segmentation runs on the acinus, not the full lobule.
# image channels: 0=actin, 1=mito, 2=lipid, 3=perox
channel_arrays = {
    "actin":  image[0],
    "mito":   image[1],
    "lipid":  image[2],
    "peroxi": image[3],
}

# Which features to compute.
# Comment out any that you don't need to (re-)compute.
feature_list = [
    "cell_mask.npy",
    "mito_mask.npy",
    "lipid_droplet_mask.npy",
    "peroxisome_mask.npy",
    "central_dist.npy",
    "portal_dist.npy",
    "boundary_dist.npy",
]

In [ ]:
# ⚠️  This cell requires a CUDA GPU.

preprocessing(
    image_path=channel_arrays,
    save_path=SAVE_PATH,
    channels=None,
    feature_list=feature_list,
)

print("Preprocessing complete!")

## 4. Inspecting segmentation results

An **instance segmentation mask** is a 2D integer array where:
- `0` = background (no object)
- `1, 2, 3, ...` = unique IDs for individual objects

This means you can count organelles simply by checking `mask.max()` (the highest ID = number of objects), or isolate a single organelle with `mask == 5`.

Let's load all four masks and visualise them overlaid on the raw image.

In [ ]:
cell_mask  = np.load(f"{SAVE_PATH}/cell_mask.npy")
mito_mask  = np.load(f"{SAVE_PATH}/mito_mask.npy")
lipid_mask = np.load(f"{SAVE_PATH}/lipid_droplet_mask.npy")
perox_mask = np.load(f"{SAVE_PATH}/peroxisome_mask.npy")

print("Mask shapes (height × width):")
print(f"  cell_mask:   {cell_mask.shape}  → {cell_mask.max()} cells")
print(f"  mito_mask:   {mito_mask.shape}  → {mito_mask.max()} mitochondria")
print(f"  lipid_mask:  {lipid_mask.shape}  → {lipid_mask.max()} lipid droplets")
print(f"  perox_mask:  {perox_mask.shape}  → {perox_mask.max()} peroxisomes")

In [ ]:
from cellpose import utils as cp_utils

def show_mask_overlay(raw_channel, mask, title, ax, raw_cmap="gray"):
    """Display a raw channel with mask outlines overlaid."""
    vmin, vmax = np.percentile(raw_channel, [1, 99])
    ax.imshow(raw_channel, cmap=raw_cmap, vmin=vmin, vmax=vmax)

    # Draw outlines of segmented objects in red
    outlines = cp_utils.masks_to_outlines(mask)
    overlay = np.zeros((*outlines.shape, 4), dtype=float)  # RGBA
    overlay[outlines == 1] = [1, 0.2, 0.2, 0.8]           # red outlines
    ax.imshow(overlay)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
show_mask_overlay(image[0], cell_mask,  "Cell mask\n(actin channel)",        axes[0], "Greens")
show_mask_overlay(image[1], mito_mask,  "Mito mask\n(mito channel)",          axes[1], "Reds")
show_mask_overlay(image[2], lipid_mask, "Lipid droplet mask\n(lipid channel)", axes[2], "Blues")
show_mask_overlay(image[3], perox_mask, "Peroxisome mask\n(perox channel)",   axes[3], "Oranges")

plt.suptitle("Segmentation masks (red outlines) overlaid on raw image", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Inspecting distance maps

### Vein distance maps

Two distance maps are computed from the vein geometry detected in the cell mask:

- `central_dist.npy` — distance (in pixels) of every pixel from the **central vein**
- `portal_dist.npy`  — distance (in pixels) of every pixel from the **portal vein**

These are used to compute the **acinus position** of each organelle:

$$\text{acinus position} = \frac{d_{\text{portal}} - d_{\text{central}}}{d_{\text{portal}} + d_{\text{central}}}$$

This normalises position to the range **[−1, +1]**, where:
- **−1** = at the central vein
- **0**  = midpoint between veins
- **+1** = at the portal vein

### Cell-boundary distance map

`boundary_dist.npy` stores, for each pixel, its distance (in pixels) to the nearest cell boundary. This is used to compute each organelle's relative position within the cell (near the edge vs. near the centre).

In [ ]:
central_dist  = np.load(f"{SAVE_PATH}/central_dist.npy")
portal_dist   = np.load(f"{SAVE_PATH}/portal_dist.npy")
boundary_dist = np.load(f"{SAVE_PATH}/boundary_dist.npy")

# Compute acinus position map for visualisation
with np.errstate(invalid="ignore"):
    acinus_pos = (portal_dist - central_dist) / (portal_dist + central_dist)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im0 = axes[0].imshow(central_dist, cmap="plasma")
axes[0].set_title("Distance from central vein (px)", fontsize=10)
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(portal_dist, cmap="plasma")
axes[1].set_title("Distance from portal vein (px)", fontsize=10)
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(acinus_pos, cmap="RdBu", vmin=-1, vmax=1)
axes[2].set_title("Acinus position\n(−1 = central vein, +1 = portal vein)", fontsize=10)
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.suptitle("Vein distance maps", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(boundary_dist, cmap="viridis")
ax.set_title("Distance from nearest cell boundary (px)", fontsize=10)
ax.axis("off")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## Summary

You have completed the preprocessing stage. The following files are now in `sample_data/output/`:

| File | Contents |
|:---|:---|
| `cell_mask.npy` | Instance segmentation of cells |
| `mito_mask.npy` | Instance segmentation of mitochondria |
| `lipid_droplet_mask.npy` | Instance segmentation of lipid droplets |
| `peroxisome_mask.npy` | Instance segmentation of peroxisomes |
| `central_dist.npy` | Distance transform from central vein |
| `portal_dist.npy` | Distance transform from portal vein |
| `boundary_dist.npy` | Distance transform from cell boundaries |

**Next:** Open `02_feature_extraction.ipynb` to extract quantitative features from these masks.